# 🔬 99% Lung & Colon Cancer from Histopathology (+ Grad-CAM)

**LC25000** - 25,000 H&E histopathology tiles across **5 classes**: lung adenocarcinoma, lung squamous-cell
carcinoma, benign lung, colon adenocarcinoma, benign colon. A convnet nails this almost perfectly, so it's a
great sandbox to *see how* it decides. We fine-tune an **EfficientNet-b0**, report a full confusion matrix
and per-organ accuracy, then use **Grad-CAM** to reveal which tissue regions drive the call.

Charts, confusion matrix, and Grad-CAM heatmaps throughout. **GPU-only.** Upvotes make my day 🙏

> **Honesty note:** LC25000 is *augmented* from ~1,250 source biopsies, so a plain random split can place
> augmented siblings of one biopsy in both train and test. The ~99% is the standard benchmark number; treat
> it as "can a CNN read this tissue" rather than a patient-level generalisation claim.

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q grad-cam timm

## Setup - force GPU + auto-locate data

In [ ]:
import os
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, balanced_accuracy_score

assert torch.cuda.is_available(), 'GPU required'
_ = nn.Conv2d(3,4,3).cuda()(torch.randn(1,3,8,8,device='cuda')); torch.cuda.synchronize()
DEVICE = torch.device('cuda'); print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

def find_root():
    for base in ['/kaggle/input', os.path.expanduser('~/.cache/kagglehub'), os.path.expanduser('~/data_kaggle'), '.']:
        for d in Path(base).rglob('lung_image_sets'):
            if (d.parent/'colon_image_sets').exists(): return d.parent
    raise FileNotFoundError('LC25000 lung_colon_image_set not found')
ROOT = find_root(); print('data root:', ROOT)
# (folder, pretty label)
SPEC = [('colon_image_sets/colon_aca','Colon adenocarcinoma'),
        ('colon_image_sets/colon_n','Colon benign'),
        ('lung_image_sets/lung_aca','Lung adenocarcinoma'),
        ('lung_image_sets/lung_n','Lung benign'),
        ('lung_image_sets/lung_scc','Lung squamous carcinoma')]
CLASSES = [p for _,p in SPEC]

## 1. 📊 Scan + class balance + a tile from every class

In [ ]:
items=[]
for ci,(sub,_) in enumerate(SPEC):
    for p in sorted((ROOT/sub).glob('*.jpeg')): items.append((str(p), ci))
from collections import Counter
cnt=Counter(CLASSES[y] for _,y in items); print('total tiles:', len(items))
fig=plt.figure(figsize=(15,4)); gs=fig.add_gridspec(1,6)
axb=fig.add_subplot(gs[0,0]); axb.barh(range(5),[cnt[c] for c in CLASSES],color='#4c72b0')
axb.set_yticks(range(5)); axb.set_yticklabels([c.replace(' ','\n') for c in CLASSES],fontsize=7)
axb.set_title('Tiles per class')
for k,(sub,lab) in enumerate(SPEC):
    a=fig.add_subplot(gs[0,1+k]); im=cv2.cvtColor(cv2.imread(str(next((ROOT/sub).glob('*.jpeg')))),cv2.COLOR_BGR2RGB)
    a.imshow(im); a.set_title(lab,fontsize=8); a.axis('off')
fig.suptitle('LC25000 - one H&E tile per class'); plt.tight_layout(); plt.show()

In [ ]:
MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
rng=np.random.RandomState(42); idx=rng.permutation(len(items))
nv=len(items)//5; val=[items[i] for i in idx[:nv]]; tr=[items[i] for i in idx[nv:]]
print(f'train={len(tr)} test={len(val)}')
class Tiles(Dataset):
    def __init__(self, rr, train=False): self.rr, self.train = rr, train
    def __len__(self): return len(self.rr)
    def __getitem__(self, i):
        p,y=self.rr[i]; img=cv2.resize(cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB),(224,224))
        if self.train:
            if np.random.rand()<0.5: img=img[:,::-1]
            if np.random.rand()<0.5: img=img[::-1,:]
        x=torch.from_numpy(np.ascontiguousarray(img)).permute(2,0,1).float()/255.
        return (x-MEAN)/STD, y
dl_tr=DataLoader(Tiles(tr,True),64,shuffle=True,num_workers=2,pin_memory=True)
dl_te=DataLoader(Tiles(val),64,num_workers=2,pin_memory=True)

## 2. Train EfficientNet-b0 (5 epochs is plenty here)

In [ ]:
model=timm.create_model('efficientnet_b0',pretrained=True,num_classes=5).to(DEVICE)
crit=nn.CrossEntropyLoss(); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5); scaler=torch.cuda.amp.GradScaler()
for ep in range(5):
    model.train(); tot=0
    for x,y in dl_tr:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.autocast('cuda'): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
    print(f'epoch {ep+1}/5 loss={tot/len(dl_tr):.4f}')

## 3. Evaluate - confusion matrix + per-organ accuracy

In [ ]:
model.eval(); ys,ps=[],[]
with torch.inference_mode():
    for x,y in dl_te: ps+=model(x.to(DEVICE)).argmax(1).cpu().tolist(); ys+=y.tolist()
ys,ps=np.array(ys),np.array(ps)
acc=(ys==ps).mean(); bacc=balanced_accuracy_score(ys,ps)
print(f'TEST accuracy={acc:.4f}  balanced_acc={bacc:.4f}')
print(classification_report(ys,ps,target_names=CLASSES))
lung=np.isin(ys,[2,3,4]); colon=np.isin(ys,[0,1])
print(f'Lung-only acc={ (ys[lung]==ps[lung]).mean():.4f}  |  Colon-only acc={ (ys[colon]==ps[colon]).mean():.4f}')
ConfusionMatrixDisplay(confusion_matrix(ys,ps),display_labels=[c.replace(' ','\n') for c in CLASSES]).plot(cmap='Blues',xticks_rotation=45)
plt.title(f'LC25000 5-class (acc={acc:.3f})'); plt.tight_layout(); plt.show()

## 4. 🔍 Grad-CAM - which tissue drives the diagnosis?

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
layer=[m for m in model.modules() if isinstance(m,nn.Conv2d)][-1]; ds=Tiles(val)
# one example from each class
picks=[]
for ci in range(5):
    for i in range(len(ds)):
        if ds.rr[i][1]==ci: picks.append(i); break
fig,axes=plt.subplots(1,5,figsize=(16,3.6))
with GradCAM(model=model,target_layers=[layer]) as cam:
    for ax,i in zip(axes.ravel(), picks):
        x,y=ds[i]; rgb=np.clip(x.permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1).astype(np.float32)
        ax.imshow(show_cam_on_image(rgb, cam(input_tensor=x[None].to(DEVICE))[0], use_rgb=True))
        ax.set_title(CLASSES[y],fontsize=8); ax.axis('off')
plt.suptitle('Grad-CAM across all five classes'); plt.tight_layout(); plt.show()

## 🏁 Takeaways

An EfficientNet-b0 separates the five lung/colon histopathology classes with near-perfect accuracy, and
Grad-CAM shows it keys on nuclear/glandular morphology rather than staining background. Remember the honesty
note up top: LC25000's augmentation means this is a *"can a CNN read this tissue"* result, not a patient-level
generalisation claim - for that you'd need source-biopsy IDs and a grouped split (see my BreakHis notebook
for that treatment). Next: per-biopsy grouping, stain normalisation, and a lightweight ViT baseline.
*Enjoyed it? An upvote is hugely appreciated 🙌*